# 日本語文字起こし 無料GPUノートブック（Kaggle / Colab）

音声・動画を **kotoba-whisper-v2.0-faster** で文字起こしする。
Whisper large-v3 と同等の日本語精度を保ったまま約6倍速く動く日本語特化モデル。

**1時間の音声が、無料のT4 GPUでおおむね2〜5分**で終わる（Intel Mac mini 2018 だと2〜5時間かかる作業）。

## 事前設定（重要）

初回のアカウント準備は [`docs/07_kaggle_setup.md`](../docs/07_kaggle_setup.md) を参照。

- **Kaggle**: 右パネル Session options → Accelerator **GPU T4 x2** / Internet **ON**（要電話番号認証・無料枠は週30時間）
  - 音声は右パネル **「+ Add Input」→「Upload」** から追加する（`/kaggle/input/` にマウントされる）
- **Colab**: ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選んで保存
  - 音声は3)のセルを実行するとダイアログが開く

GPUを付け忘れるとCPUで動いてしまい、10倍以上遅くなる。1セル目の表示で必ず確認すること。
Kaggleは Internet が OFF だとモデルをダウンロードできず4)で失敗する。

## 使い方
上から順にセルを実行 → 音声を読み込み → 文字起こし → zipをダウンロード。
初回はモデルのダウンロード（約1.5GB）で数分かかる。**複数ファイルをまとめて処理すると枠の節約になる。**

対応形式: mp3 / wav / m4a / flac / mp4 / mkv など（動画はそのまま入れてよい）

> このノートブックはGPUの無い環境で作成したため実行未検証。
> エラーが出た場合はセルの出力メッセージを添えて相談してください。

In [ ]:
# 1) GPU環境の確認
import torch

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    DEVICE, COMPUTE = 'cuda', 'float16'
    print(f'GPU: {torch.cuda.get_device_name(0)} / VRAM {vram_gb:.1f} GB')
else:
    DEVICE, COMPUTE = 'cpu', 'int8'
    print('GPUが見えていません。CPUでも動きますが10倍以上遅くなります。')
    print('Colab: ランタイム→ランタイムのタイプを変更→T4 GPU')
    print('Kaggle: Session options→Accelerator→GPU T4 x2')

In [ ]:
# 2) faster-whisper のインストール（1〜2分）
!pip install -q faster-whisper

import ctranslate2, faster_whisper
print('faster-whisper', getattr(faster_whisper, '__version__', '?'),
      '/ ctranslate2', ctranslate2.__version__)

# libcudnn_ops_infer.so.8 が無い等のcuDNNエラーで落ちる場合は、
# 次の行のコメントを外して実行し、ランタイムを再起動してから1)からやり直す:
# !pip install -q "ctranslate2==4.4.0"

In [ ]:
# 3) 音声・動画の読み込み（KaggleとColabで置き場所が違うので自動判別する）
import os, glob

IS_KAGGLE = os.path.exists('/kaggle/working')
WORK = '/kaggle/working' if IS_KAGGLE else '/content'
OUT_DIR = os.path.join(WORK, 'transcripts')
os.makedirs(OUT_DIR, exist_ok=True)

EXTS = ('.mp3', '.wav', '.m4a', '.flac', '.ogg', '.opus', '.aac',
        '.mp4', '.mkv', '.mov', '.webm')

if IS_KAGGLE:
    # 右パネル「+ Add Input」→「Upload」で音声をアップロードすると
    # /kaggle/input/<データセット名>/ に読み取り専用でマウントされる
    targets = sorted(p for p in glob.glob('/kaggle/input/**/*', recursive=True)
                     if os.path.isfile(p) and p.lower().endswith(EXTS))
    if not targets:
        print('/kaggle/input に音声が見つかりません。')
        print('右パネルの「+ Add Input」→「Upload」から音声を追加してください。')
else:
    AUDIO_DIR = os.path.join(WORK, 'audio')
    os.makedirs(AUDIO_DIR, exist_ok=True)
    try:
        from google.colab import files  # ダイアログが開く（複数選択可）
        for name, data in files.upload().items():
            with open(os.path.join(AUDIO_DIR, name), 'wb') as f:
                f.write(data)
    except ImportError:
        print(f'{AUDIO_DIR} に音声を置いてから、このセルを再実行してください')
    targets = sorted(p for p in glob.glob(os.path.join(AUDIO_DIR, '*'))
                     if os.path.isfile(p))

print(f'対象 {len(targets)}件:', [os.path.basename(p) for p in targets])

# 1GBを超えるような大きいファイルはブラウザ経由が遅い。Colabならドライブ経由が速い:
# from google.colab import drive; drive.mount('/content/drive')
# targets = sorted(glob.glob('/content/drive/MyDrive/録音/*'))

In [ ]:
# 4) モデルの読み込み（初回はダウンロードで数分）
from faster_whisper import WhisperModel

# 日本語特化・large-v3並みの精度で約6倍速。日本語の音声ならこれが第一候補
MODEL_ID = 'kotoba-tech/kotoba-whisper-v2.0-faster'
# 英語や他言語が混ざる音声なら次に差し替える（多言語対応の汎用モデル）:
# MODEL_ID = 'large-v3-turbo'

try:
    model = WhisperModel(MODEL_ID, device=DEVICE, compute_type=COMPUTE)
except Exception as e:
    print('GPUでの読み込みに失敗 → CPU int8 で再試行します:', e)
    DEVICE, COMPUTE = 'cpu', 'int8'
    model = WhisperModel(MODEL_ID, device=DEVICE, compute_type=COMPUTE)

print(f'{MODEL_ID} を {DEVICE}/{COMPUTE} で読み込みました')

In [ ]:
# 5) 文字起こし実行 → .txt（本文）と .srt（字幕・タイムコード付き）を書き出す
import os, time

# 固有名詞のヒント。キャラ名・番組名などを書いておくと表記ゆれが減る（120文字程度まで）
HINT = ''


def srt_time(sec):
    h, rem = divmod(int(sec), 3600)
    m, s = divmod(rem, 60)
    return f'{h:02d}:{m:02d}:{s:02d},{int((sec % 1) * 1000):03d}'


# chunk_length / condition_on_previous_text はkotoba-whisperのモデルカード推奨値。
# 長い音声で同じ文が延々繰り返される事故を防ぐ設定でもある
OPTS = dict(language='ja', beam_size=5, vad_filter=True,
            chunk_length=15, condition_on_previous_text=False,
            initial_prompt=HINT or None)

for path in targets:
    base = os.path.splitext(os.path.basename(path))[0]
    print(f'--- {base} ---')
    t0 = time.time()
    try:
        segments, info = model.transcribe(path, **OPTS)
    except TypeError:  # 古いfaster-whisperには chunk_length が無い
        segments, info = model.transcribe(
            path, **{k: v for k, v in OPTS.items() if k != 'chunk_length'})

    lines, srt, i = [], [], 0
    for i, seg in enumerate(segments, 1):  # ここで実際の推論が進む
        text = seg.text.strip()
        lines.append(text)
        srt.append(f'{i}\n{srt_time(seg.start)} --> {srt_time(seg.end)}\n{text}\n')
        if i % 20 == 0:
            print(f'  {i}セグメント / 音声{seg.end / 60:.1f}分まで完了')

    with open(os.path.join(OUT_DIR, base + '.txt'), 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines) + '\n')
    with open(os.path.join(OUT_DIR, base + '.srt'), 'w', encoding='utf-8') as f:
        f.write('\n'.join(srt))

    dur, el = getattr(info, 'duration', 0.0), time.time() - t0
    print(f'完了: {i}セグメント / 音声{dur / 60:.1f}分 を {el:.0f}秒で処理'
          f' (実時間の約{dur / max(el, 1):.0f}倍速)')

In [ ]:
# 6) 結果をまとめてダウンロード
import shutil, os

zip_path = shutil.make_archive(os.path.join(WORK, 'transcripts_out'), 'zip', OUT_DIR)
print(zip_path, f'({os.path.getsize(zip_path) / 1024:.0f} KB)')
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print(f'Kaggleの場合: 右パネルの {zip_path} をダウンロードしてください')

## うまくいかないときは

| 症状 | 対処 |
|---|---|
| 1)で「GPUが見えていません」 | ランタイム設定でT4 GPUを選び直し、ランタイムを再起動して1)からやり直す |
| `libcudnn_ops_infer.so.8` が無いと出る | 2)のコメント行 `ctranslate2==4.4.0` を有効化 → ランタイム再起動 → 1)から |
| 同じ文が延々と繰り返される | 無音や音楽が長い区間で起きやすい。`vad_filter=True` のままか確認。改善しなければ `beam_size=1` を試す |
| 固有名詞がずっと違う字になる | 5)の `HINT` にキャラ名などを書いて再実行 |
| 途中でセッションが切れた | 無料枠の上限。音声を30分程度に分割して回すと安全 |

## 手元のPCで動かしたい場合

同じモデルをローカルでも使える。詳しくは `docs/06_transcription.md` を参照。

```bash
pip install faster-whisper
```

- **配信PC（Windows / RTX 3050）**: `device='cuda', compute_type='float16'` でそのまま動く。1時間の音声が5〜10分程度
- **Mac mini 2018（Intel）**: `device='cpu', compute_type='int8'` に変更。10分程度の音声までが実用範囲